# 01 - Data Exploration

Load the international results dataset using `src/data.py` and take a first look at it.

In [ ]:
import sys

sys.path.append("..")

from src.data import load_results, run_sanity_checks

In [ ]:
df = load_results()
run_sanity_checks(df)

In [ ]:
df.head()

In [ ]:
df.describe(include="all")

## Elo ratings

Compute leakage-free, rolling Elo ratings for every team using `src/elo.py`, then
look at the top 20 teams by current (most recent) rating.

In [ ]:
from src.elo import compute_elo_ratings, top_teams

df_elo, current_ratings = compute_elo_ratings(df)
df_elo[["date", "home_team", "away_team", "home_elo_pre", "away_elo_pre"]].tail()

In [ ]:
top_teams(current_ratings, n=20)

## Validate the K-factor mapping

Data-quality check (no changes to `elo.py` here): look at how `get_k_factor()`
classifies the `tournament` values that actually occur in results.csv, and flag
any high-frequency tournament that lands in the generic `K_OTHER` bucket but
arguably deserves a higher tier (i.e. its name doesn't exactly match anything
in `MAJOR_TOURNAMENTS`, even though it is a competitive, high-stakes
tournament).

In [ ]:
from src.elo import K_OTHER, get_k_factor

tournament_counts = df["tournament"].value_counts().rename_axis("tournament").reset_index(name="count")
tournament_counts["k_factor"] = tournament_counts["tournament"].apply(get_k_factor)
tournament_counts

In [ ]:
TOP_N = 30

top_tournaments = tournament_counts.head(TOP_N)
print(f"K-factor assigned to the {TOP_N} most common tournaments:")
top_tournaments

In [ ]:
# Among the most common tournaments, which ones land in the generic K_OTHER
# bucket (i.e. didn't match FIFA World Cup, MAJOR_TOURNAMENTS, "qualification",
# or Friendly)?
other_bucket = top_tournaments[top_tournaments["k_factor"] == K_OTHER]
print(f"High-frequency tournaments (top {TOP_N}) mapped to the generic K_OTHER bucket:")
other_bucket

In [ ]:
# Flag entries in that bucket whose name suggests a competitive, high-stakes
# tournament (e.g. a confederation "Nations League") that just doesn't happen
# to match the exact strings in MAJOR_TOURNAMENTS.
flag_keywords = ["Nations League"]
flagged = other_bucket[
    other_bucket["tournament"].str.contains("|".join(flag_keywords), case=False)
]

print("Flagged for review -- arguably major, but currently mapped to K_OTHER:")
print(flagged[["tournament", "count", "k_factor"]].to_string(index=False))


## Knockout advancement demo

Fit the goals model on all available results, then use `src/knockout.py` to turn
those match-level goal rates into P(advances) for five round-of-16 ties (all
neutral venue). Elo ratings used are the latest available snapshot from
`current_ratings` -- for the tie whose match has already been played in
results.csv (France-Paraguay), this is a retrospective sanity check rather
than a blind forecast, since that Elo snapshot already reflects the actual
result.

In [ ]:
import pandas as pd

from src.model import fit_poisson_model
from src.knockout import advance_probability

as_of = df_elo["date"].max() + pd.Timedelta(days=1)
goals_model = fit_poisson_model(df_elo, as_of_date=as_of)
print(f"Fitted on {goals_model.n_matches_used} matches (rho={goals_model.rho:.3f}, converged={goals_model.converged})")

In [ ]:
round_of_16_ties = [
    ("Argentina", "Egypt"),
    ("Spain", "Portugal"),
    ("Brazil", "Norway"),
    ("Mexico", "England"),
    ("France", "Paraguay"),
]

rows = []
for team_a, team_b in round_of_16_ties:
    result = advance_probability(
        goals_model,
        team_a,
        team_b,
        current_ratings.get(team_a, 1500.0),
        current_ratings.get(team_b, 1500.0),
        neutral=True,
    )
    rows.append(
        {
            "team_a": team_a,
            "team_b": team_b,
            "p_a_advances": result.p_a_advances,
            "p_b_advances": result.p_b_advances,
            "p_a_win_90": result.p_a_win_90,
            "p_draw_90": result.p_draw_90,
            "p_level_after_et": result.p_level_after_et,
        }
    )

pd.DataFrame(rows)

## Betting odds demo (OddsPapi)

Fetch pre-game 1X2 odds for one already-played round-of-32 fixture (Mexico vs
Ecuador) and print the de-vigged market-implied probabilities. Requires
`ODDSPAPI_API_KEY` set in a `.env` file -- this cell skips gracefully if it
isn't set, and once a fixture is cached under `data/odds_cache/`, re-running
this cell costs zero API requests. Since the match has already been played,
`get_fixture_market_probabilities` falls back to the closing pre-kickoff line
from `/v4/historical-odds` (see `src/odds.py` docstring for why that
endpoint's *last* snapshot isn't automatically the right one to use).

In [ ]:
from src.odds import has_api_key, find_world_cup_fixtures, find_fixture_id_by_teams, get_fixture_market_probabilities

if not has_api_key():
    print("ODDSPAPI_API_KEY is not set -- add it to a .env file to run this demo.")
else:
    finished_fixtures = find_world_cup_fixtures(status_id=2)  # 2 = finished
    fixture_id = find_fixture_id_by_teams(finished_fixtures, "Mexico", "Ecuador")

    odds = get_fixture_market_probabilities(fixture_id)
    print(f"{odds.home_team} vs {odds.away_team} ({odds.date}) -- {odds.source}")
    print(f"decimal odds: home={odds.home_odds} draw={odds.draw_odds} away={odds.away_odds}")
    print(
        f"de-vigged market probabilities: "
        f"P(home)={odds.p_home:.1%}  P(draw)={odds.p_draw:.1%}  P(away)={odds.p_away:.1%}"
    )

## Backtest: model vs market on already-played knockout fixtures

⚠️ **This is an approximation, not a fair comparison -- see the "fair 90-minute
backtest" section further down for why, and for the honest side-by-side
numbers.** `src/evaluation.py` walks forward through every already-played
knockout fixture (Round of 32 onward), refitting the goals model as-of each
fixture's own date so nothing about that fixture (or anything later) leaks
into its own prediction. Ties are resolved to an actual winner by checking
which team reappears in a later fixture (see `_resolve_actual_winner` --
results.csv doesn't record extra-time/penalty winners directly). The market's
P(advance) is de-vigged Pinnacle 1X2 odds converted via
`P(win) + 0.5 * P(draw)` -- a crude coin-flip approximation, since there is no
real market for "wins the tie" to use instead. This takes a couple of
minutes: one goals-model fit per unique fixture date, plus one OddsPapi
lookup per fixture (mostly served from cache after the first run).

In [ ]:
from src.evaluation import backtest_knockout_fixtures

comparison, summary = backtest_knockout_fixtures(df)
comparison

In [ ]:
print(f"Fixtures scored: {summary['n_fixtures']} (skipped: {summary['n_skipped']})")
print(f"Brier score  -- model: {summary['model_brier']:.4f}  market: {summary['market_brier']:.4f}")
print(f"Log loss     -- model: {summary['model_log_loss']:.4f}  market: {summary['market_log_loss']:.4f}")
print(f"Model beat market on {summary['n_model_beats_market']}/{summary['n_fixtures']} fixtures")

## Live predictions: round-of-16 fixtures not yet played

Predict the remaining round-of-16 ties with the goals model fit on every
available result, and compare against the market's current de-vigged
P(advance). `generate_live_predictions` auto-detects unplayed fixtures in the
round-of-16 date window from results.csv, but also drops any fixture whose
OddsPapi market has already closed -- results.csv is a live feed and can lag
reality, so a fixture can still show a missing score there after the market
(and the real match) has already moved on. That's exactly what happened here:
of the 6 fixtures results.csv still shows as unplayed, Brazil-Norway and
Mexico-England turned out to already have a closed ("closing line") market
and were dropped, leaving the 4 genuinely still-open ties. The predictions
are saved with a generation timestamp to `predictions/round_of_16_live.json`
as a verifiable pre-match record.

In [ ]:
from src.evaluation import (
    ROUND_OF_16_START_DATE,
    ROUND_OF_16_END_DATE,
    find_unplayed_fixtures_in_window,
    generate_live_predictions,
    save_predictions_json,
)

unplayed = find_unplayed_fixtures_in_window(df, ROUND_OF_16_START_DATE, ROUND_OF_16_END_DATE)
live_records, live_model, live_as_of = generate_live_predictions(df, unplayed)

save_predictions_json(live_records, "../predictions/round_of_16_live.json", live_as_of)
pd.DataFrame(live_records)

## Model calibration check: Elo vs. market

A concrete symptom surfaced a calibration bug: the model gave France only ~32%
to beat Morocco in regulation despite a ~150-point Elo edge, against the
market's ~60%. Root cause: Morocco's small-sample recent-form defense
parameter was swamping the (well-calibrated) Elo term. The fix -- raising
`reg_strength` (L2 shrinkage on the attack/defense parameters) from 0.01 to
10.0 -- was chosen by backtesting against real outcomes (see
`backtest_knockout_fixtures` above and `src/model.py`'s "Calibration finding"
docstring section), *not* by fitting to the market, per the brief.

The cell below reproduces France-Morocco plus the four round-of-16 ties at
both the old and new `reg_strength`, against the market, and reports the mean
absolute gap.

In [ ]:
from src.model import fit_poisson_model, predict_match
from src.odds import find_world_cup_fixtures, get_market_probabilities_for_teams

calibration_fixtures = [
    ("France", "Morocco", True),
    ("Portugal", "Spain", True),
    ("United States", "Belgium", False),
    ("Argentina", "Egypt", True),
    ("Switzerland", "Colombia", True),
]

odds_fixtures_all = find_world_cup_fixtures()

rows = []
for reg_strength in [0.01, 10.0]:
    calib_model = fit_poisson_model(df_elo, as_of_date=as_of, reg_strength=reg_strength)
    for home, away, neutral in calibration_fixtures:
        pred = predict_match(
            calib_model, home, away,
            current_ratings.get(home, 1500.0), current_ratings.get(away, 1500.0),
            neutral=neutral,
        )
        market = get_market_probabilities_for_teams(odds_fixtures_all, home, away)
        if market is None:
            continue
        rows.append(
            {
                "reg_strength": reg_strength,
                "home_team": home,
                "away_team": away,
                "model_p_home_win": pred.home_win,
                "market_p_home_win": market.p_home,
                "abs_gap": abs(pred.home_win - market.p_home),
            }
        )

calibration_df = pd.DataFrame(rows)
calibration_df

In [ ]:
print("Mean absolute |model P(home win 90') - market P(home win 90')| gap, by reg_strength:")
print(calibration_df.groupby("reg_strength")["abs_gap"].mean())

## Elo blend: correcting favorite bias

Even after the `reg_strength` fix above, the Poisson model stays conservative
on clear favorites: it gives France only ~39% to beat Morocco in regulation
(a ~150-point Elo edge), against the market's ~60%. `src/blend.py` adds a
principled ensemble -- `P_final = w * P_poisson + (1 - w) * P_elo` -- where
`P_elo` comes from the classical Elo expected-score formula plus a
data-derived draw model (see that module's docstring for the exact
derivation). **This is a bias correction validated against real outcomes
below, not an attempt to clone the market** -- the market is used here only
as a diagnostic to see the *effect* of a candidate weight.

The cell below sweeps `blend_weight` over 0.5-1.0 for France-Morocco plus the
four round-of-16 ties and reports the mean absolute gap to market.

In [ ]:
blend_rows = []
for blend_weight in [0.5, 0.6, 0.7, 0.8, 1.0]:
    for home, away, neutral in calibration_fixtures:
        pred = predict_match(
            goals_model, home, away,
            current_ratings.get(home, 1500.0), current_ratings.get(away, 1500.0),
            neutral=neutral, blend_weight=blend_weight,
        )
        market = get_market_probabilities_for_teams(odds_fixtures_all, home, away)
        if market is None:
            continue
        blend_rows.append(
            {
                "blend_weight": blend_weight,
                "home_team": home,
                "away_team": away,
                "model_p_home_win": pred.home_win,
                "market_p_home_win": market.p_home,
                "abs_gap": abs(pred.home_win - market.p_home),
            }
        )

blend_df = pd.DataFrame(blend_rows)
blend_df

In [ ]:
print("Mean absolute |model P(home win) - market P(home win)| gap, by blend_weight:")
print(blend_df.groupby("blend_weight")["abs_gap"].mean())
print()
print(
    "blend_weight=0.7 (the default) keeps the Poisson model as the primary signal "
    "(70% weight) while capturing nearly all of the available gap reduction -- "
    "going lower would move closer to the market but at the cost of trusting Elo "
    "over the goals model more than intended."
)

## Fair backtest: model vs market on the 90-minute result

The "advance" backtest above has a methodological asymmetry: the model runs
its full extra-time/penalty machinery, while the market side is only ever a
crude `P(win) + 0.5*P(draw)` coin-flip approximation, since there's no real
market for "wins the tie." Part of any apparent model edge there could just
be the model doing more work on a question the market was never asked.

`backtest_90min_fixtures` is the fair, apples-to-apples alternative: it scores
model vs market on exactly what the market prices pre-match -- the 90-minute
1X2 result (home win / draw / away win) -- with no knockout-stage logic on
either side, using the multiclass Brier score and log loss (sum/negative-log
over all three outcomes, not just one). We report both backtests side by
side, not one in place of the other.

In [ ]:
from src.evaluation import backtest_90min_fixtures

comparison_90, summary_90 = backtest_90min_fixtures(df)
comparison_90

In [ ]:
print("=== Side by side (model vs market, both at blend_weight=1.0 i.e. pure Poisson) ===")
print(
    f"Advance (APPROXIMATION) -- model Brier={summary['model_brier']:.4f}  "
    f"market Brier={summary['market_brier']:.4f}  |  "
    f"model log loss={summary['model_log_loss']:.4f}  market log loss={summary['market_log_loss']:.4f}  |  "
    f"model beat market {summary['n_model_beats_market']}/{summary['n_fixtures']}"
)
print(
    f"90-minute (FAIR)        -- model Brier={summary_90['model_brier']:.4f}  "
    f"market Brier={summary_90['market_brier']:.4f}  |  "
    f"model log loss={summary_90['model_log_loss']:.4f}  market log loss={summary_90['market_log_loss']:.4f}  |  "
    f"model beat market {summary_90['n_model_beats_market']}/{summary_90['n_fixtures']}"
)
print()
print(
    "Note: small, in-tournament sample either way (n < 25 fixtures) -- read both "
    "as an early signal, not a claim of long-run edge over the market. On the fair "
    "90-minute comparison at pure Poisson (blend_weight=1.0, shown above -- the same "
    "setting used for the 'advance' comparison, to isolate the fairness question from "
    "the blend-tuning question above), the market currently outperforms the model, "
    "which is the expected result for a sharp book like Pinnacle against a "
    "from-scratch model this early in the tournament."
)
print()
print(
    "For reference: the deployed app (scripts/update_data.py, data/app/backtest_90min.json) "
    "uses the Elo-blended model (blend_weight=0.7) for this same fair comparison, where the "
    "blend's bias correction is enough to edge narrowly ahead of the market on Brier score -- "
    "see the Track Record tab."
)